In [2]:
import os

# You can only use one GPU at a time
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
os.environ["OMP_NUM_THREADS"] = "16"
os.environ["MKL_NUM_THREADS"] = "16"

In [3]:
from vv.metrics.cmmd import CMMDMetric
from tqdm import tqdm

./venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import torch

# Define the shape of the images (e.g., 3 channels, 224x224 resolution)
image_shape = (3, 256, 256)
batch_size = 15
num_batches = 10

# Generate random image-shaped tensors
#batches_1 = [torch.rand(batch_size, *image_shape) for _ in range(num_batches)]
batches_1 = [torch.ones(batch_size, *image_shape) for _ in range(num_batches)]
batches_2 = [torch.rand(batch_size, *image_shape) for _ in range(num_batches)]
batches_3 = [torch.rand(batch_size, *image_shape) for _ in range(num_batches)]

In [6]:
cmmd_metric = CMMDMetric()

./venv/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `Clip MMD` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)


In [7]:
# Move tensors to CUDA
batches_1 = [batch.to('cuda') for batch in batches_1]
batches_2 = [batch.to('cuda') for batch in batches_2]
batches_3 = [batch.to('cuda') for batch in batches_3]

# Move embedding_model of CMMDMetric to CUDA
cmmd_metric.to('cuda')

CMMDMetric(
  (embedding_model): ClipEmbeddingModel(
    (_model): CLIPVisionModelWithProjection(
      (vision_model): CLIPVisionTransformer(
        (embeddings): CLIPVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
          (position_embedding): Embedding(577, 1024)
        )
        (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (encoder): CLIPEncoder(
          (layers): ModuleList(
            (0-23): 24 x CLIPEncoderLayer(
              (self_attn): CLIPSdpaAttention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
              )
              (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=Tr

In [8]:
for i, (batch_1, batch_2) in tqdm(enumerate(zip(batches_1, batches_2))):
    # Compute the CMMD metric for the current batch
    cmmd_metric.update(batch_1, batch_2)

10it [00:08,  1.25it/s]


In [9]:
out = cmmd_metric.compute()
# Print the CMMD metric with more digits
print("CMMD metric:", f"{out.item():.10f}")
print(type(out))

CMMD metric: 5.6844949722
<class 'torch.Tensor'>


./venv/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: MMD computation might not be reliable with 10 samples.
  warnings.warn(*args, **kwargs)
./venv/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: The state is a list, indicating that the tensors have not been reduced yet. Concatenating all tensors to compute the final MMDIn a distributed setting, this may lead to incorrect reduction.Please ensure the strategy calls the sync() function of the metric.
  warnings.warn(*args, **kwargs)


In [8]:
cmmd_metric.reset()

out = cmmd_metric.compute()
print("CMMD metric after reset:", f"{out.item():.10f}")
print(type(out))

./venv/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: The ``compute`` method of metric CMMDMetric was called before the ``update`` method which may lead to errors, as metric states have not yet been updated.
  warnings.warn(*args, **kwargs)


RuntimeError: Cannot compute MMD: no embeddings were provided.